# Notebook 8: Predictor de la matriz de covarianzas

**Objetivo del notebook:** construir un modelo que, a partir del
comportamiento reciente de los 30 activos, prediga su matriz de
covarianzas (el riesgo conjunto) en un periodo futuro. Calcular la matriz
de covarianzas de un periodo que ya ha ocurrido es aritmética directa
(`np.cov`, usada en el Paso 1 para construir el objetivo de cada ejemplo
de entrenamiento); el problema real que resuelve este notebook es
distinto: predecir el riesgo de un periodo que todavía no ha pasado, para
el cual no existen datos reales con los que calcularlo directamente. Eso
requiere un modelo que aprenda el patrón "así se comportó el pasado
reciente -> así resultó el riesgo después" a partir de muchos ejemplos
históricos, y lo aplique a una ventana reciente cuyo futuro real aún se
desconoce.

## Paso 1: Construcción de ejemplos de entrenamiento

Cada ejemplo tiene la forma general:
- **Entrada (X):** retornos de los 30 activos durante un periodo histórico
  (`history_days`).
- **Objetivo (y):** matriz de covarianzas REAL de los 30 activos, calculada
  con lo que efectivamente ocurrió en el periodo siguiente (`horizon_days`).

El tamaño de ambos periodos no está decidido de antemano: en la sección 2
se exploran varias combinaciones (no solo la propuesta inicial de 1 año de
historia para predecir 2 meses) antes de fijar cuál se usa para continuar
el resto del notebook.

La ventana se desliza cada `step_days` días para generar múltiples
ejemplos a partir del histórico completo. Tampoco se fija un único valor
de `step_days` de entrada: se explora junto con las combinaciones de
`history_days`/`horizon_days` en la sección 2, ya que un paso pequeño
genera más ejemplos pero muy solapados entre sí, y uno grande genera menos
ejemplos pero más independientes.

Se trabaja sobre los retornos reales de los 30 activos (Notebook 1), no
sobre los componentes descompuestos (Notebook 2) — la descomposición en
factores era necesaria para los generadores (poder simular shocks), pero el
predictor final trabaja directamente sobre los retornos, tal como se
planteó desde el principio.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.predictor.windowing import build_covariance_windows, summarize_windows

## 1. Cargar los retornos reales (Notebook 1)

In [ ]:
returns_real = pd.read_parquet("../data/processed/returns_daily.parquet")
print(f"Shape: {returns_real.shape}")
print(f"Rango de fechas: {returns_real.index.min()} -> {returns_real.index.max()}")

## 2. Construir las ventanas (histórico -> Sigma futura)

**Antes de fijar una única configuración, exploramos varias combinaciones**
de tamaño de entrada (historia), horizonte a predecir, y también del paso
entre ejemplos (`step_days`) — no solo history/horizon, para no asumir un
valor de step_days a ciegas. Un `step_days` pequeño da más ejemplos, pero
muy solapados entre sí (casi el mismo dato repetido); uno grande da menos
ejemplos, pero más independientes.

In [ ]:
# Combinaciones de (history_days, horizon_days) a probar
combinaciones_horizonte = [
    (126, 21),   # entrada 6 meses -> predecir 1 mes
    (252, 42),   # entrada 1 año -> predecir 2 meses (propuesta inicial del planteamiento)
    (504, 42),   # entrada 2 años -> predecir 2 meses
    (252, 21),   # entrada 1 año -> predecir 1 mes
]

# Valores de step_days a probar, cruzados con cada combinación anterior.
# Se incluye 1 ("cada día es un ejemplo nuevo", el mismo enfoque usado en los generadores)
steps_a_probar = [1, 5, 21, 63]

exploracion = []
for history_days, horizon_days in combinaciones_horizonte:
    for step_days in steps_a_probar:
        w = build_covariance_windows(
            returns_real, history_days=history_days, horizon_days=horizon_days, step_days=step_days
        )
        exploracion.append({
            "history_days": history_days,
            "horizon_days": horizon_days,
            "step_days": step_days,
            "n_ejemplos": len(w["X"]),
        })

pd.DataFrame(exploracion)

**Nota:** esta celda solo compara cuántos ejemplos genera cada combinación
(un chequeo rápido de viabilidad, ahora incluyendo `step_days`). La
comparación real de qué combinación predice mejor solo se puede hacer una
vez montado el modelo (Paso 3), entrenando con cada una y comparando el
error de predicción — eso se hará más adelante, no en este paso.

Para poder seguir construyendo el resto de este notebook (sanity check,
comprobación de validez, y después el modelo) hace falta trabajar con una
única combinación como referencia, en vez de repetir cada paso muchas
veces. Se usa `history_days=252, horizon_days=42, step_days=21` (1 año de
historia para predecir 2 meses, la propuesta inicial del planteamiento)
solo para avanzar; no es una decisión cerrada. Las demás combinaciones
(incluyendo distintos `step_days`) quedan anotadas y se compararán de
verdad en el Paso 3, cuando el modelo ya esté entrenado y se pueda medir
el error de predicción de cada una con datos reales.

In [ ]:
HISTORY_DAYS = 252   # ~1 año de bolsa
HORIZON_DAYS = 42    # ~2 meses de bolsa
STEP_DAYS = 21       # ~1 mes entre ejemplos consecutivos

windows = build_covariance_windows(
    returns_real,
    history_days=HISTORY_DAYS,
    horizon_days=HORIZON_DAYS,
    step_days=STEP_DAYS,
)

summary = summarize_windows(windows, n_assets=returns_real.shape[1])
summary

## 3. Sanity check de un ejemplo concreto

Revisamos el primer ejemplo para confirmar que las fechas y los shapes
tienen sentido antes de seguir.

In [ ]:
i = 0
print(f"Fecha de corte del ejemplo {i}: {windows['cutoff_dates'][i]}")
print(f"Shape de X (ventana histórica): {windows['X'][i].shape}")
print(f"Shape de y (Sigma futura): {windows['y'][i].shape}")
print()
print("Primeras filas de la Sigma real de este ejemplo:")
pd.DataFrame(
    windows["y"][i][:5, :5],
    index=returns_real.columns[:5],
    columns=returns_real.columns[:5],
)

## 4. Comprobación de validez: ¿son las Sigma reales matrices válidas?

Antes de seguir, comprobamos que todas las matrices de covarianzas
"reales" (las que sirven de objetivo) cumplen la propiedad de ser
semidefinidas positivas (ningún autovalor negativo) — deberían cumplirla
siempre, al ser calculadas directamente con `np.cov` sobre datos reales,
pero conviene confirmarlo antes de construir el modelo.

In [ ]:
min_eigenvalues = [np.linalg.eigvalsh(sigma).min() for sigma in windows["y"]]

print(f"Autovalor mínimo encontrado (de todos los ejemplos): {min(min_eigenvalues):.2e}")
print(f"Número de ejemplos con algún autovalor negativo: {sum(e < 0 for e in min_eigenvalues)}")

## 5. Resultado del Paso 1

Con la configuración de referencia (252 días de historia, 42 de horizonte,
paso de 21 días) se generaron 119 ejemplos, cubriendo desde el 5 de enero
de 2016 hasta el 11 de noviembre de 2025. El rango no cubre los 10 años
completos del histórico porque cada ejemplo consume 252 días hacia atrás
(para la ventana de entrada) y 42 hacia adelante (para poder calcular la
Sigma objetivo); los extremos del histórico no tienen suficiente margen en
una de las dos direcciones y quedan excluidos.

Todas las matrices Sigma reales resultaron válidas (semidefinidas
positivas): el autovalor mínimo encontrado en el conjunto completo fue
positivo (6.09e-07) y ningún ejemplo presentó autovalores negativos —
resultado esperado, al calcularse directamente con `np.cov` sobre datos
reales, pero es la comprobación que garantiza que el objetivo de
entrenamiento (`y`) es siempre correcto antes de construir el modelo.

**Siguiente paso (Paso 2):** decidir la representación de salida del
modelo (descomposición de Cholesky), para garantizar que cualquier
predicción de la red sea automáticamente una matriz de covarianzas válida,
sin depender de que la red "aprenda" esa restricción por sí sola.

## Paso 2: Representación de salida (Cholesky)

El modelo tiene que predecir una matriz de covarianzas — pero no es una
tabla de números cualquiera: tiene que cumplir una regla matemática
obligatoria ("semidefinida positiva"), que en la práctica significa que el
riesgo calculado para cualquier combinación posible de los 30 activos
nunca puede salir negativo. No es una regla sobre cada número por
separado — de hecho, valores individuales negativos en la matriz son
normales y válidos (indican que dos activos tienden a moverse en
direcciones opuestas, como se vio en el sanity check del Paso 1, ej.
B-AAPL). Es una regla sobre cómo se relacionan TODOS los números entre sí
a la vez, así que no basta con prohibir a la red que prediga negativos:
eso ni garantiza la regla real (que depende de la combinación conjunta,
no de valores sueltos) ni sería siquiera deseable (impediría representar
relaciones inversas reales entre activos).

Si le pedimos a la red que prediga directamente los ~465 valores únicos de
Sigma (30x30 por simetría), no hay ninguna garantía de que el conjunto
cumpla esa regla: la red los ajustaría uno a uno intentando minimizar el
error, sin ningún mecanismo que le impida romper la propiedad global.

La solución (descomposición de Cholesky) evita el problema por
construcción: en lugar de predecir Sigma directamente, la red predice un
vector de parámetros completamente libres, sin restricciones. Con ese
vector se construye una matriz auxiliar L (triangular inferior), y se
calcula `Sigma = L @ L.T` (L multiplicada por su propia traspuesta). Esta
multiplicación concreta garantiza matemáticamente que el resultado cumple
la regla, sea cual sea el contenido de L — la restricción queda
incorporada en la operación fija que se aplica después de la predicción,
no en algo que la red deba aprender por sí sola.

Un matiz adicional: los elementos de la diagonal de L deben ser
estrictamente positivos para que la garantía sea completa (Sigma definida
positiva, no solo semidefinida — la propiedad más fuerte, y la que
interesa en la práctica). Para lograrlo sin restringir tampoco esos
valores, la red predice su logaritmo (`log(L_ii)`, un número libre) y se
recupera el valor real aplicando la exponencial — la exponencial de
cualquier número, sea el que sea, da siempre un resultado positivo.

In [ ]:
from src.predictor.cholesky import (
    sigma_to_cholesky_vector,
    cholesky_vector_to_sigma,
    cholesky_vector_length,
)

n_assets = returns_real.shape[1]
vector_length = cholesky_vector_length(n_assets)

print(f"Número de activos: {n_assets}")
print(f"Longitud del vector de parámetros libres que predecirá la red: {vector_length}")
print(f"(frente a los {n_assets * n_assets} valores de la matriz completa, o los "
      f"{n_assets * (n_assets - 1) // 2 + n_assets} valores únicos por simetría)")

### Comprobación de ida y vuelta

Sobre las Sigma reales del Paso 1: se descompone cada una en su vector de
Cholesky y se reconstruye de vuelta, comprobando que se recupera la matriz
original (sin pérdida de información) y que el resultado reconstruido sigue
siendo una matriz válida.

In [ ]:
max_reconstruction_diff = 0.0

for sigma_real in windows["y"]:
    vector = sigma_to_cholesky_vector(sigma_real)
    sigma_reconstructed = cholesky_vector_to_sigma(vector, n=n_assets)

    diff = np.abs(sigma_real - sigma_reconstructed).max()
    max_reconstruction_diff = max(max_reconstruction_diff, diff)

print(f"Diferencia máxima de reconstrucción (sobre los {len(windows['y'])} ejemplos): "
      f"{max_reconstruction_diff:.2e}")

## Resultado del Paso 2

Para los 30 activos, el vector de parámetros libres tiene longitud 465 —
el mismo número que los valores únicos de Sigma por simetría, pero sin
restricciones sobre ellos (frente a los 900 valores de la matriz completa
sin explotar la simetría). La diferencia máxima de reconstrucción sobre
los 119 ejemplos del Paso 1 fue de 3.47e-18 — del orden de la precisión
numérica del ordenador, es decir, esencialmente cero. Esto confirma que la
transformación Cholesky no pierde información: cualquier matriz de
covarianzas válida se puede representar exactamente como un vector de
parámetros libres, y ese vector se puede convertir de vuelta a la matriz
original sin pérdida.

Esta representación depende únicamente del número de activos (30), no de
la configuración de ventanas temporales del Paso 1 (historia, horizonte,
paso entre ejemplos): son dos decisiones independientes que no interactúan
entre sí, y cualquier combinación que se explore más adelante en el Paso 3
usará esta misma representación de salida sin cambios.

Esta es la representación de salida que usará la red en el Paso 3: en
lugar de predecir Sigma directamente, predice el vector de longitud
`n_assets*(n_assets+1)/2`, y la conversión a Sigma (mediante
`cholesky_vector_to_sigma`) se aplica siempre después, como parte fija del
pipeline, no como algo que el modelo deba aprender.

**Siguiente paso (Paso 3):** construir y entrenar la red (candidatos: GRU,
LSTM, y un baseline simple sin red neuronal), usando datos reales.

## Paso 3: Construcción y entrenamiento del predictor

Se comparan tres candidatos:
1. **Baseline simple, sin red neuronal:** usa directamente la covarianza
   de la propia ventana de entrada como predicción ("el riesgo futuro se
   parece al riesgo reciente"). Cualquier modelo con red neuronal debería
   superar este resultado para justificar su complejidad.
2. **GRU**
3. **LSTM**

Los tres se evalúan con la misma métrica: el error cuadrático medio entre
la matriz predicha y la matriz real, promediado sobre todas sus casillas.

**Split temporal:** los ejemplos se dividen en entrenamiento y test
respetando el orden cronológico (nunca al azar) — el test siempre queda
compuesto por los ejemplos más recientes, para simular cómo se usaría el
modelo en la práctica (solo con información pasada disponible). Dentro del
conjunto de entrenamiento, se reserva además una porción (también la más
reciente dentro de ese subconjunto) como validación para el early
stopping — el test queda completamente al margen del entrenamiento en
todo momento.

**Normalización:** se ajusta un `StandardScaler` (media y desviación por
activo) únicamente con los datos de entrenamiento, y se aplica igual a
entrenamiento, validación y test — nunca se ajusta con datos de test, para
no filtrar información del futuro al proceso de normalización.

**Presupuesto de entrenamiento:** máximo 1000 épocas, paciencia 150 (el
mismo acuerdo aplicado a los modelos generativos).

In [ ]:
import torch

from src.predictor.split import temporal_split_purged, fit_scaler, apply_scaler
from src.predictor.baseline import baseline_predict, evaluate_predictions
from src.predictor.models import RecurrentCovariancePredictor, init_output_bias_from_data
from src.predictor.train import train_predictor
from src.predictor.cholesky import cholesky_vector_length

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")

### 3.1 Split temporal con purga (train / validación / test)

**Ajuste importante respecto a la primera versión:** con `step_days=21`
solo se generaban 119 ejemplos en total (76 para entrenar). Al comentar
esto, se aclaró que el solapamiento entre ejemplos NO es un problema en sí
mismo — es la práctica habitual al generar ventanas de series temporales
(se hace también en el VAE, con `step=1`, "cada día que pasa es un ejemplo
nuevo") — así que se puede bajar `step_days` para tener muchos más
ejemplos, sin ningún problema, siempre que se cuide la frontera entre
entrenamiento y test.

Donde sí hay riesgo real de fuga de información es en esa frontera: si un
ejemplo de entrenamiento y uno de test comparten casi todos sus días
históricos (por estar justo pegados en el tiempo), el modelo estaría
prácticamente "viendo" el test durante el entrenamiento. Se soluciona con
una **zona de purga** en esa frontera (la misma técnica, `PURGE_SIZE`, que
se usa en el VAE): se descartan los últimos ejemplos de entrenamiento que
queden dentro de la distancia de solapamiento (`history_days +
horizon_days`) respecto al primer ejemplo de test.

Se regeneran las ventanas con `step_days=1` (frente al 21 anterior),
fiel al mismo enfoque usado en el VAE (Notebook 5), "cada día que pasa es
un ejemplo nuevo".

In [ ]:
STEP_DAYS = 1  # antes: 21 — se baja a 1 ("cada día es un ejemplo nuevo"), el mismo enfoque usado en el VAE

windows = build_covariance_windows(
    returns_real, history_days=HISTORY_DAYS, horizon_days=HORIZON_DAYS, step_days=STEP_DAYS,
)
print(f"Total de ejemplos con step_days={STEP_DAYS}: {len(windows['X'])} (antes, con step_days=21: 119)")

In [ ]:
# Primero: train vs test (20%, más reciente), con purga en la frontera
split_test = temporal_split_purged(
    windows, test_fraction=0.2,
    history_days=HISTORY_DAYS, horizon_days=HORIZON_DAYS, step_days=STEP_DAYS,
)
print(f"Ejemplos purgados en la frontera train/test: {split_test['n_purged']}")

# Del train (ya purgado), se reserva una porción para validación, con su propia purga
windows_train_only = {
    "X": split_test["X_train"],
    "y": split_test["y_train"],
    "cutoff_dates": split_test["dates_train"],
}
split_val = temporal_split_purged(
    windows_train_only, test_fraction=0.2,
    history_days=HISTORY_DAYS, horizon_days=HORIZON_DAYS, step_days=STEP_DAYS,
)
print(f"Ejemplos purgados en la frontera train/validación: {split_val['n_purged']}")

X_train, y_train = split_val["X_train"], split_val["y_train"]
X_val, y_val = split_val["X_test"], split_val["y_test"]
X_test, y_test = split_test["X_test"], split_test["y_test"]

print(f"Entrenamiento: {len(X_train)} ejemplos ({split_val['dates_train'][0]} -> {split_val['dates_train'][-1]})")
print(f"Validación:    {len(X_val)} ejemplos ({split_val['dates_test'][0]} -> {split_val['dates_test'][-1]})")
print(f"Test:          {len(X_test)} ejemplos ({split_test['dates_test'][0]} -> {split_test['dates_test'][-1]})")

### 3.2 Normalización

In [ ]:
scaler = fit_scaler(X_train)

X_train_scaled = apply_scaler(X_train, scaler)
X_val_scaled = apply_scaler(X_val, scaler)
X_test_scaled = apply_scaler(X_test, scaler)

### 3.3 Baseline (sin red neuronal)

Se evalúa sobre el conjunto de test (sin normalizar: el baseline no usa
ninguna red, así que trabaja directamente con los retornos originales).

In [ ]:
baseline_preds_test = baseline_predict(X_test)
baseline_results = evaluate_predictions(baseline_preds_test, y_test)
baseline_results

### 3.4 GRU

**Ajuste — calibración del punto de partida de la red:** en una primera
prueba, la red predecía Sigma en una escala completamente equivocada —
por ejemplo, una diagonal media de 6.70e-02 frente a la real de 3.65e-04
(unas 180 veces mayor). Con solo 76 ejemplos de entrenamiento, la red
tenía muy poco margen para corregir por sí sola un punto de partida tan
alejado de la escala real: al inicializarse con pesos aleatorios, y al
recuperarse la diagonal de L mediante una exponencial (ver
`cholesky_torch.py`), la primera predicción podía caer en cualquier orden
de magnitud, muy lejos de la escala diminuta típica de retornos
financieros diarios (varianzas del orden de 1e-4).

La solución (`init_output_bias_from_data`) calibra el sesgo de la capa de
salida ANTES de empezar a entrenar, usando la varianza media real de los
datos de entrenamiento: pone a cero los pesos de la capa de salida y fija
el sesgo de la parte logarítmica de la diagonal a
`0.5*log(varianza_media_real)`, de forma que la primera predicción de la
red, antes de cualquier entrenamiento, ya tenga aproximadamente la escala
correcta — el entrenamiento entonces solo tiene que afinar los detalles
(las relaciones entre activos), en lugar de tener que descubrir también
la escala completa desde cero con muy pocos datos disponibles.

In [ ]:
n_assets = returns_real.shape[1]
output_dim = cholesky_vector_length(n_assets)

torch.manual_seed(42)
gru_model = RecurrentCovariancePredictor(
    n_assets=n_assets, output_dim=output_dim, cell_type="gru", hidden_size=16, num_layers=1, dropout=0.2,
)
init_output_bias_from_data(gru_model, y_train, n_assets)

gru_result = train_predictor(
    gru_model, X_train_scaled, y_train, X_val_scaled, y_val,
    n_assets=n_assets, max_epochs=1000, patience=150, device=DEVICE,
)

print(f"GRU — mejor época: {gru_result['best_epoch']}, val_loss: {gru_result['best_val_loss']:.6e}")

In [ ]:
history_df = pd.DataFrame(gru_result["history"])
history_df.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(10, 4), title="Curva de entrenamiento — GRU")
plt.axvline(gru_result["best_epoch"], color="green", linestyle="--", alpha=0.6, label="mejor época")
plt.legend()
plt.show()

### 3.5 LSTM

In [ ]:
torch.manual_seed(42)
lstm_model = RecurrentCovariancePredictor(
    n_assets=n_assets, output_dim=output_dim, cell_type="lstm", hidden_size=16, num_layers=1, dropout=0.2,
)
init_output_bias_from_data(lstm_model, y_train, n_assets)

lstm_result = train_predictor(
    lstm_model, X_train_scaled, y_train, X_val_scaled, y_val,
    n_assets=n_assets, max_epochs=1000, patience=150, device=DEVICE,
)

print(f"LSTM — mejor época: {lstm_result['best_epoch']}, val_loss: {lstm_result['best_val_loss']:.6e}")

In [ ]:
history_df_lstm = pd.DataFrame(lstm_result["history"])
history_df_lstm.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(10, 4), title="Curva de entrenamiento — LSTM")
plt.axvline(lstm_result["best_epoch"], color="green", linestyle="--", alpha=0.6, label="mejor época")
plt.legend()
plt.show()

### 3.6 Evaluación final sobre test (los tres candidatos)

In [ ]:
from src.predictor.cholesky_torch import cholesky_vector_to_sigma_torch

def predict_sigmas(model, X_scaled, n_assets, device):
    model.eval()
    X_t = torch.tensor(np.stack(X_scaled), dtype=torch.float32, device=device)
    with torch.no_grad():
        vectors = model(X_t)
        sigmas = cholesky_vector_to_sigma_torch(vectors, n=n_assets)
    return [s.cpu().numpy() for s in sigmas]

gru_preds_test = predict_sigmas(gru_model, X_test_scaled, n_assets, DEVICE)
lstm_preds_test = predict_sigmas(lstm_model, X_test_scaled, n_assets, DEVICE)

gru_results = evaluate_predictions(gru_preds_test, y_test)
lstm_results = evaluate_predictions(lstm_preds_test, y_test)

comparison = pd.DataFrame([
    {"modelo": "Baseline (persistencia)", **baseline_results},
    {"modelo": "GRU", **gru_results, "mejor_epoca": gru_result["best_epoch"]},
    {"modelo": "LSTM", **lstm_results, "mejor_epoca": lstm_result["best_epoch"]},
])
comparison

### 3.7 Diagnóstico: ¿por qué el baseline gana con tanta diferencia?

El baseline supera claramente a GRU y LSTM (varios órdenes de magnitud), y
"mejor época = 1000" en ambas redes indica que el early stopping nunca se
activó — el modelo seguía "mejorando" en validación hasta agotar el
presupuesto completo. Antes de seguir ajustando hiperparámetros a ciegas,
comparamos una predicción concreta de la red contra la Sigma real
correspondiente, para ver si el problema es de escala (la red predice
números en un orden de magnitud equivocado) o de estructura (la escala es
correcta pero las relaciones entre activos no se capturan bien).

In [ ]:
i = 0  # primer ejemplo de test

print("--- Sigma REAL (5x5) ---")
print(y_test[i][:5, :5])
print()
print("--- Sigma predicha por la GRU (5x5) ---")
print(gru_preds_test[i][:5, :5])
print()
print(f"Escala típica de la Sigma real (media de la diagonal): {np.diag(y_test[i]).mean():.2e}")
print(f"Escala típica de la Sigma predicha (media de la diagonal): {np.diag(gru_preds_test[i]).mean():.2e}")

**Interpretación:** si la escala predicha (media de la diagonal) es
muchos órdenes de magnitud distinta de la escala real, el problema es que
la red no está ajustando bien la magnitud de los valores (posible causa:
el `learning_rate` o la inicialización no son adecuados para valores tan
pequeños como los retornos diarios). Si las escalas son parecidas pero los
valores concretos difieren mucho entre sí, el problema es más bien de
estructura (la red no está capturando bien las relaciones entre activos).

### 3.8 Matriz de comparación: todas las combinaciones de ventanas

Con la calibración de escala (sección 3.4) el `mejor_epoca` pasó de 1000 a
5-7 — la red converge casi de inmediato porque ya no tiene que descubrir
la escala, pero eso también significa que con solo 76 ejemplos de
entrenamiento, el modelo agota rápido lo poco que puede aprender y deja de
mejorar, lejos del entrenamiento largo y gradual que cabría esperar con
más datos.

Se ejecuta el pipeline completo (split con purga, escalado, baseline, GRU,
LSTM) para TODAS las combinaciones de `history_days`/`horizon_days`/
`step_days` anotadas en el Paso 1 — incluyendo, en particular, valores de
`step_days` pequeños (1 y 5), que generan muchos más ejemplos de
entrenamiento al solapar más las ventanas. El split usa `temporal_split_purged`
(sección 3.1), que elimina los ejemplos de entrenamiento demasiado cercanos
a la frontera con validación/test, para que el solapamiento entre ejemplos
(que no es problema en sí mismo) no se traduzca en fuga de información
hacia la evaluación.

In [ ]:
from src.predictor.pipeline import run_all_configs

todas_las_combinaciones = [
    (history_days, horizon_days, step_days)
    for history_days, horizon_days in combinaciones_horizonte
    for step_days in steps_a_probar
]

resultados_todas_config = run_all_configs(
    returns_real, todas_las_combinaciones,
    hidden_size=16, dropout=0.2, max_epochs=1000, patience=150, device=DEVICE,
)
resultados_todas_config

**Lectura de la tabla:** buscar filas donde `gru_best_epoch` o
`lstm_best_epoch` sean sustancialmente mayores que 5-7 (indicando un
entrenamiento más largo y gradual), y comparar `baseline_mse` contra
`gru_mse`/`lstm_mse` en cada configuración (¿alguna red llega a superar
al baseline con más datos disponibles?). La configuración elegida como
definitiva para el resto del proyecto será la que combine el mejor
resultado de test con un entrenamiento que demuestre convergencia gradual
genuina, no solo el menor error aislado.

### 3.9 Visualización de la tabla comparativa (análisis interno)

**Nota:** estas dos gráficas son solo para facilitar la lectura de la
tabla de 16 filas y decidir la configuración final — no sustituyen a las
curvas de entrenamiento (train_loss/val_loss por época) de las secciones
3.4 y 3.5, que son las que muestran la convergencia real del modelo.

In [ ]:
resultados_validos = resultados_todas_config[resultados_todas_config["error"].isna()] \
    if "error" in resultados_todas_config.columns else resultados_todas_config

resultados_validos = resultados_validos.copy()
resultados_validos["config_label"] = resultados_validos.apply(
    lambda r: f"h{int(r['history_days'])}_hz{int(r['horizon_days'])}_s{int(r['step_days'])}", axis=1
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(resultados_validos))
width = 0.25

ax.bar(x - width, resultados_validos["baseline_mse"], width, label="Baseline")
ax.bar(x, resultados_validos["gru_mse"], width, label="GRU")
ax.bar(x + width, resultados_validos["lstm_mse"], width, label="LSTM")

ax.set_xticks(x)
ax.set_xticklabels(resultados_validos["config_label"], rotation=90)
ax.set_ylabel("MSE de test")
ax.set_title("MSE por configuración (barras más bajas = mejor)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(resultados_validos["n_train"], resultados_validos["gru_best_epoch"], label="GRU", alpha=0.7)
ax.scatter(resultados_validos["n_train"], resultados_validos["lstm_best_epoch"], label="LSTM", alpha=0.7)
ax.set_xlabel("n_train (ejemplos de entrenamiento)")
ax.set_ylabel("mejor_epoca")
ax.set_title("¿Más ejemplos de entrenamiento -> convergencia más gradual?")
ax.legend()
plt.tight_layout()
plt.show()

## Paso 4 (versión final): búsqueda + entrenamiento con el modelo combinado

Combina 3 cambios acumulativos respecto al Paso 3:
1. **Purga en el split** (ya aplicada desde el Paso 3 en adelante).
2. **Tamaño de activos variable** (Paso 4, arquitectura de mini-red por activo).
3. **Conexión residual al baseline**: en vez de que la fórmula de factores
   sustituya la predicción, se SUMA al baseline —
   `Sigma_final = Sigma_baseline + (F @ F.T + diag(idio²))`. La suma de
   dos matrices semidefinidas positivas sigue siendo semidefinida
   positiva, así que la garantía de validez se mantiene. Con la red
   inicializada cerca de cero (`init_factor_encoder_near_baseline`), la
   predicción inicial coincide casi exactamente con el baseline (mismo
   principio que en la sección 3.10, adaptado a esta arquitectura).

**Por qué se repite la búsqueda de configuración de ventanas:** la
configuración `history_days=504, horizon_days=42` se eligió en el Paso 3
usando el modelo GRU/LSTM simple — no hay garantía de que sea también la
mejor para esta arquitectura combinada, distinta de la que se usó para
elegirla. Se repite la búsqueda específicamente con el modelo final.

**Ajuste práctico de alcance:** el entrenamiento aquí es ejemplo a
ejemplo (no en lotes), mucho más lento que en el Paso 3. Repetir la
búsqueda completa de 16 combinaciones no es viable en tiempo (se probó
`step_days=1` con esta arquitectura y una sola configuración tardó más de
9 horas sin terminar). Por eso:
- Se fija `step_days=5` (no se re-explora: ya se comprobó en el Paso 3 que
  `step_days=1` es inviable en tiempo con este tipo de entrenamiento, y
  `step_days=5` fue consistentemente de las configuraciones con mejor
  resultado en la búsqueda original).
- Se prueban solo 4 combinaciones de `history_days`/`horizon_days` (las
  mismas 4 de la exploración del Paso 3), no las 16 completas.
- La fase de búsqueda usa un presupuesto de épocas reducido (solo para
  comparar configuraciones entre sí); la configuración ganadora se
  reentrena después con más épocas para el resultado final.

In [ ]:
from src.predictor.search_combined import search_combined_configs, run_combined_config

combinaciones_busqueda = [(126, 21), (252, 42), (504, 42), (252, 21)]

resultados_combinados = search_combined_configs(
    returns_real, combinaciones_busqueda, n_assets=n_assets,
    search_max_epochs=30, search_patience=10,
    n_subsets_per_example=5, n_factors=4, hidden_size=16, dropout=0.25,
    device=DEVICE,
)
resultados_combinados

**Lectura:** elegir la configuración con `model_mse` más cercano (o menor)
a `baseline_mse`, con un `n_train` razonable (no extremadamente bajo).

In [ ]:
# Ajustar estos valores según la fila ganadora de la tabla anterior
BEST_HISTORY = 504
BEST_HORIZON = 42

final_result = run_combined_config(
    returns_real, BEST_HISTORY, BEST_HORIZON, step_days=5, n_assets=n_assets,
    max_epochs=150, patience=30,
    n_subsets_per_example=5, n_factors=4, hidden_size=16, dropout=0.25,
    device=DEVICE,
)
print(f"Configuración final: history={BEST_HISTORY}, horizon={BEST_HORIZON}, step=5")
print(f"Ejemplos: train={final_result['n_train']}, val={final_result['n_val']}, test={final_result['n_test']}")
print(f"Baseline MSE: {final_result['baseline_mse']:.4e}")
print(f"Modelo combinado MSE: {final_result['model_mse']:.4e}")
print(f"Mejor época: {final_result['best_epoch']}")

In [ ]:
final_model = final_result["model"]

# El historial detallado no se recoge en run_combined_config (para mantenerlo
# simple); si se quiere la curva de esta configuración ganadora, reentrenar
# directamente con train_predictor_factor_residual guardando el historial:
from src.predictor.factor_model import AssetFactorEncoder, init_factor_encoder_near_baseline
from src.predictor.train_factor import train_predictor_factor_residual

data = final_result["data"]
torch.manual_seed(42)
final_model_with_history = AssetFactorEncoder(n_factors=4, cell_type="gru", hidden_size=16, dropout=0.25)
init_factor_encoder_near_baseline(final_model_with_history, baseline_train=data["baseline_train"])

final_training = train_predictor_factor_residual(
    final_model_with_history, data["X_train"], data["y_train"], data["baseline_train"],
    data["X_val"], data["y_val"], data["baseline_val"],
    max_epochs=150, patience=30, device=DEVICE, verbose_every=10,
    history_csv_path="../data/processed/factor_model_final_history.csv",
)

history_df_final = pd.DataFrame(final_training["history"])
history_df_final.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(10, 4), title="Curva de entrenamiento — modelo final combinado")
plt.axvline(final_training["best_epoch"], color="green", linestyle="--", alpha=0.6, label="mejor época")
plt.legend()
plt.show()

## Resultado del Paso 3-4 y conclusión

**Resumen de todo lo probado, de más simple a más elaborado:**

1. **GRU/LSTM simple** (30 activos fijos, Paso 3): resultado cerca del
   baseline sin superarlo, convergencia corta (3-9 épocas) pero limpia.
2. **Conexión residual** sobre el modelo simple (sección 3.10): mejoró el
   MSE relativo, pero la convergencia larga observada (145-437 épocas)
   resultó ser oscilación sin aprendizaje genuino tras una inspección
   detallada de la curva, no una mejora real.
3. **Tamaño de activos variable**, sin residual (primer intento del Paso
   4): el mejor MSE relativo hasta ese momento (prácticamente empatado
   con el baseline), pero de nuevo con convergencia corta.
4. **Modelo combinado** (tamaño variable + residual, esta sección):
   mismo patrón que las anteriores — cerca del baseline sin superarlo de
   forma sostenida, con mejor época muy temprana incluso dando margen de
   sobra (`patience=30`), pese a la búsqueda de configuración repetida
   específicamente para esta arquitectura.

**Conclusión general del proceso:** en todas las variantes probadas, el
modelo se queda cerca del baseline de persistencia sin superarlo de forma
clara y sostenida. Esto es un resultado honesto y consistente, no un
fallo de una configuración concreta: sugiere que, con la cantidad de datos
disponible, el riesgo financiero reciente es difícil de mejorar como
predictor mediante estas arquitecturas. Esa pregunta se aborda de forma
sistemática en el Paso 5 (datos sintéticos): si añadir los datasets
generados por los cuatro generadores del proyecto ayuda a cerrar esa
diferencia.

## Paso 5: ¿ayudan los datos sintéticos?

Pregunta abierta que dejó la conclusión del Paso 3-4: con datos reales,
ningún modelo probado supera al baseline de forma clara — ¿cambia eso si
se añaden los datos sintéticos generados en el Notebook 7 (ruido,
Gaussiana, VAE, autorregresivo, cada uno usado según su diseño)?

Se reutiliza exactamente la misma arquitectura y configuración que ganó
en el Paso 4 (`AssetFactorEncoder`, `dropout=0.25`, conexión residual al
baseline, `max_epochs=150, patience=30`) — la única variable que cambia
entre entrenamientos es qué datos ve el modelo durante el entrenamiento.
Validación y test son siempre los mismos ejemplos 100% reales (guardados
en el Notebook 7), para que las seis variantes sean directamente
comparables entre sí.

In [ ]:
from pathlib import Path
from src.predictor.factor_model import AssetFactorEncoder, init_factor_encoder_near_baseline
from src.predictor.train_factor import train_predictor_factor_residual, predict_sigma_factor_residual

COMBINED_DIR = Path("../data/processed/combined_datasets")

val_test = np.load(COMBINED_DIR / "validacion_y_test_real.npz")
X_val_p5 = list(val_test["X_val"])
y_val_p5 = list(val_test["y_val"])
X_test_p5 = list(val_test["X_test"])
y_test_p5 = list(val_test["y_test"])

baseline_val_p5 = baseline_predict(X_val_p5)
baseline_test_p5 = baseline_predict(X_test_p5)
baseline_mse_p5 = evaluate_predictions(baseline_test_p5, y_test_p5)["mse_medio"]
print(f"Baseline MSE (test, compartido por las 6 variantes): {baseline_mse_p5:.4e}")

DATASETS_PASO5 = ["solo_real", "real_mas_noise", "real_mas_gaussian", "real_mas_vae", "real_mas_ar", "real_mas_todo"]

resultados_paso5 = []
modelos_paso5 = {}

for nombre_dataset in DATASETS_PASO5:
    data = np.load(COMBINED_DIR / f"{nombre_dataset}.npz")
    X_train_p5, y_train_p5 = list(data["X"]), list(data["y"])
    baseline_train_p5 = baseline_predict(X_train_p5)

    torch.manual_seed(42)
    modelo_p5 = AssetFactorEncoder(n_factors=4, cell_type="gru", hidden_size=16, dropout=0.25)
    init_factor_encoder_near_baseline(modelo_p5, baseline_train=baseline_train_p5)

    resultado = train_predictor_factor_residual(
        modelo_p5, X_train_p5, y_train_p5, baseline_train_p5,
        X_val_p5, y_val_p5, baseline_val_p5,
        max_epochs=150, patience=30, device=DEVICE,
    )

    preds_test_p5 = [
        predict_sigma_factor_residual(modelo_p5, x, b, device=DEVICE)
        for x, b in zip(X_test_p5, baseline_test_p5)
    ]
    model_mse_p5 = evaluate_predictions(preds_test_p5, y_test_p5)["mse_medio"]

    resultados_paso5.append({
        "dataset": nombre_dataset, "n_train": len(X_train_p5),
        "baseline_mse": baseline_mse_p5, "model_mse": model_mse_p5,
        "mejor_epoca": resultado["best_epoch"],
    })
    modelos_paso5[nombre_dataset] = modelo_p5
    print(f"{nombre_dataset:22s} n_train={len(X_train_p5):3d}  model_mse={model_mse_p5:.4e}  mejor_epoca={resultado['best_epoch']}")

resultados_paso5_df = pd.DataFrame(resultados_paso5)
resultados_paso5_df

**Lectura:** comparar `model_mse` contra `baseline_mse` (la misma en las
6 filas, es el mismo test real) — ¿alguna variante baja de forma clara por
debajo del baseline? `solo_real` debería salir muy parecido al resultado
ya visto en el Paso 4, como comprobación de consistencia.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(resultados_paso5_df))
ax.bar(x, resultados_paso5_df["model_mse"], color="C0", label="Modelo")
ax.axhline(baseline_mse_p5, color="crimson", linestyle="--", label="Baseline (test real)")
ax.set_xticks(x)
ax.set_xticklabels(resultados_paso5_df["dataset"], rotation=30, ha="right")
ax.set_ylabel("MSE de test")
ax.set_title("Paso 5 — MSE de test según qué datos vieron el modelo en entrenamiento")
ax.legend()
plt.tight_layout()
plt.show()

## Conclusión del Paso 5

Los seis datasets de entrenamiento (solo real, y cinco variantes
aumentadas con sintéticos) dan un resultado prácticamente idéntico:

| Dataset | n_train | MSE modelo | Mejor época |
|---|---|---|---|
| Baseline (test) | — | 6.4392e-08 | — |
| solo_real | 88 | 6.4343e-08 | 3 |
| real_mas_noise | 176 | 6.4336e-08 | 2 |
| real_mas_gaussian | 176 | 6.4336e-08 | 2 |
| real_mas_vae | 176 | 6.4343e-08 | 2 |
| real_mas_ar | 264 | 6.4342e-08 | 2 |
| real_mas_todo | 528 | 6.4338e-08 | 1 |

Las seis variantes caen en el mismo rango, a un pelo del baseline
(diferencias en el 4º-5º decimal — ruido, no señal). Ni siquiera con 528
ejemplos (6 veces más que solo con datos reales) la mejor época pasa de
1-3: más datos, sean reales o sintéticos de cualquiera de los 4
generadores, no le dieron al modelo ningún margen adicional para
aprender.

**Respuesta a la pregunta central del proyecto:** con esta arquitectura y
esta cantidad de datos reales de partida, añadir datos sintéticos no
mejora el predictor de covarianzas. No es un resultado negativo por un
fallo de ejecución — se probaron los 4 generadores del proyecto, cada uno
según su diseño (ruido y Gaussiana como series completas, autorregresivo
generando futuros alternativos, VAE editando tramos de la historia), con
una metodología sin fugas de información (validación y test siempre 100%
reales) y una comparación justa (mismo test para las 6 variantes). El
techo parece estar en la propia tarea — predecir covarianza a 42 días,
donde la persistencia de la volatilidad a corto plazo ya es difícil de
batir — y no en la falta de datos ni en la calidad de los generadores.

## Apéndice: experimentos intermedios

**Nota:** las secciones 3.10 y el Paso 4 original que siguen a continuación
fueron experimentos intermedios. La versión final (tamaño de activos
variable + conexión residual, combinados) está en el **Paso 4 (versión
final)**, más arriba, con su propia búsqueda de configuración. Estas
secciones se conservan como registro del proceso, no como código a
ejecutar de nuevo.

### 3.10 Conexión residual al baseline (experimento intermedio, ver nota arriba)

**Idea:** en lugar de que la red prediga la Sigma
completa desde cero, que prediga solo una CORRECCIÓN sobre el baseline —
la red recibe la ventana de retornos y predice un ajuste, que se suma al
vector Cholesky del baseline antes de reconstruir la Sigma final. Con la
capa de salida inicializada a cero (`init_output_layer_to_zero`), la
corrección inicial es nula: la primera predicción de la red, antes de
cualquier entrenamiento, coincide EXACTAMENTE con el baseline (comprobado:
diferencia de ~4e-11, precisión numérica). Esto da una garantía natural —
en el peor caso, el modelo iguala al baseline; en el mejor caso, aprende a
mejorarlo.

A diferencia de la calibración de escala del Paso 3.4
(`init_output_bias_from_data`, que estimaba la escala real a partir de los
datos porque la red no tenía ninguna pista), aquí ya no hace falta: el
baseline aporta la escala correcta directamente.

Se prueba sobre la configuración con mejor equilibrio del Paso 3.9
(`history_days=504, horizon_days=42`), comparando `step_days=5` (más
cercano a los resultados de convergencia larga observados) frente al
resultado sin conexión residual de la misma configuración.

In [ ]:
from src.predictor.baseline import baseline_cholesky_vectors
from src.predictor.models import init_output_layer_to_zero
from src.predictor.train import train_predictor_residual, predict_sigmas_residual

HISTORY_DAYS_BEST = 504
HORIZON_DAYS_BEST = 42
STEP_DAYS_BEST = 5

windows_best = build_covariance_windows(
    returns_real, history_days=HISTORY_DAYS_BEST, horizon_days=HORIZON_DAYS_BEST, step_days=STEP_DAYS_BEST,
)

split_test_best = temporal_split_purged(
    windows_best, test_fraction=0.2,
    history_days=HISTORY_DAYS_BEST, horizon_days=HORIZON_DAYS_BEST, step_days=STEP_DAYS_BEST,
)
windows_train_only_best = {
    "X": split_test_best["X_train"], "y": split_test_best["y_train"], "cutoff_dates": split_test_best["dates_train"],
}
split_val_best = temporal_split_purged(
    windows_train_only_best, test_fraction=0.2,
    history_days=HISTORY_DAYS_BEST, horizon_days=HORIZON_DAYS_BEST, step_days=STEP_DAYS_BEST,
)

X_train_b, y_train_b = split_val_best["X_train"], split_val_best["y_train"]
X_val_b, y_val_b = split_val_best["X_test"], split_val_best["y_test"]
X_test_b, y_test_b = split_test_best["X_test"], split_test_best["y_test"]

scaler_b = fit_scaler(X_train_b)
X_train_b_scaled = apply_scaler(X_train_b, scaler_b)
X_val_b_scaled = apply_scaler(X_val_b, scaler_b)
X_test_b_scaled = apply_scaler(X_test_b, scaler_b)

baseline_train_vecs = baseline_cholesky_vectors(X_train_b)
baseline_val_vecs = baseline_cholesky_vectors(X_val_b)
baseline_test_vecs = baseline_cholesky_vectors(X_test_b)

baseline_preds_b = baseline_predict(X_test_b)
baseline_mse_b = evaluate_predictions(baseline_preds_b, y_test_b)["mse_medio"]

print(f"Ejemplos: train={len(X_train_b)}, val={len(X_val_b)}, test={len(X_test_b)}")
print(f"Baseline MSE (esta configuración): {baseline_mse_b:.4e}")

In [ ]:
torch.manual_seed(42)
gru_residual = RecurrentCovariancePredictor(
    n_assets=n_assets, output_dim=output_dim, cell_type="gru", hidden_size=16, dropout=0.2,
)
init_output_layer_to_zero(gru_residual)

gru_residual_result = train_predictor_residual(
    gru_residual, X_train_b_scaled, y_train_b, baseline_train_vecs,
    X_val_b_scaled, y_val_b, baseline_val_vecs,
    n_assets=n_assets, max_epochs=1000, patience=150, device=DEVICE,
)
print(f"GRU residual — mejor época: {gru_residual_result['best_epoch']}, val_loss: {gru_residual_result['best_val_loss']:.6e}")

torch.manual_seed(42)
lstm_residual = RecurrentCovariancePredictor(
    n_assets=n_assets, output_dim=output_dim, cell_type="lstm", hidden_size=16, dropout=0.2,
)
init_output_layer_to_zero(lstm_residual)

lstm_residual_result = train_predictor_residual(
    lstm_residual, X_train_b_scaled, y_train_b, baseline_train_vecs,
    X_val_b_scaled, y_val_b, baseline_val_vecs,
    n_assets=n_assets, max_epochs=1000, patience=150, device=DEVICE,
)
print(f"LSTM residual — mejor época: {lstm_residual_result['best_epoch']}, val_loss: {lstm_residual_result['best_val_loss']:.6e}")

In [ ]:
history_df_gru_res = pd.DataFrame(gru_residual_result["history"])
history_df_gru_res.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(10, 4), title="Curva de entrenamiento — GRU residual")
plt.axvline(gru_residual_result["best_epoch"], color="green", linestyle="--", alpha=0.6, label="mejor época")
plt.legend()
plt.show()

In [ ]:
history_df_lstm_res = pd.DataFrame(lstm_residual_result["history"])
history_df_lstm_res.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(10, 4), title="Curva de entrenamiento — LSTM residual")
plt.axvline(lstm_residual_result["best_epoch"], color="green", linestyle="--", alpha=0.6, label="mejor época")
plt.legend()
plt.show()

In [ ]:
gru_residual_preds = predict_sigmas_residual(gru_residual, X_test_b_scaled, baseline_test_vecs, n_assets, DEVICE)
lstm_residual_preds = predict_sigmas_residual(lstm_residual, X_test_b_scaled, baseline_test_vecs, n_assets, DEVICE)

gru_residual_mse = evaluate_predictions(gru_residual_preds, y_test_b)["mse_medio"]
lstm_residual_mse = evaluate_predictions(lstm_residual_preds, y_test_b)["mse_medio"]

comparison_residual = pd.DataFrame([
    {"modelo": "Baseline", "mse": baseline_mse_b, "mejor_epoca": None},
    {"modelo": "GRU residual", "mse": gru_residual_mse, "mejor_epoca": gru_residual_result["best_epoch"]},
    {"modelo": "LSTM residual", "mse": lstm_residual_mse, "mejor_epoca": lstm_residual_result["best_epoch"]},
])
comparison_residual

**Lectura:** comparar `gru_residual_mse`/`lstm_residual_mse` contra `baseline_mse_b` de esta misma
configuración — gracias a la inicialización a cero, en el peor de los casos deberían quedar muy cerca
del baseline (nunca mucho peor), y si el entrenamiento encuentra alguna corrección útil, deberían
mejorarlo. Para comparar contra la versión SIN conexión residual de esta misma configuración
(`history_days=504, horizon_days=42, step_days=5`), consultar la fila correspondiente en la tabla
de la sección 3.8 (`resultados_todas_config`).

## Paso 4 (primer intento, sin residual) — experimento intermedio, ver nota arriba

**Motivación:** todo lo anterior está atado a un
número fijo de activos (los 30 completos). Aunque las secciones 3.1-3.10
generaron más ejemplos (más fechas, más solapamiento), seguían siendo
siempre "el mismo grupo fijo de 30 activos" — poca variedad genuina de
información, aunque la cantidad bruta de ejemplos subiera.

**El cambio de arquitectura:** en lugar de una red que mira todos los
activos a la vez, se usa una mini-red que procesa UN activo cada vez (sin
necesitar saber nada de los demás), aplicada tantas veces como activos
tenga cada ejemplo. Para cada activo predice un vector de "carga
factorial" y un componente idiosincrático — la misma lógica conceptual
que la descomposición del Notebook 2 (factor común + idiosincrático), pero
aprendida por la red en lugar de calculada con regresión, y sin depender
de que el grupo de activos sea siempre el mismo.

**Reconstrucción de Sigma:** `Sigma = F @ F.T + diag(idio²)`, donde F es
la matriz de cargas factoriales (una fila por activo). Esta fórmula
garantiza que el resultado sea siempre una matriz de covarianzas válida,
para cualquier número de activos — es la versión de tamaño variable de la
garantía que daba Cholesky en el Paso 2 (que, al depender su longitud de
vector del número de activos, solo servía para un tamaño fijo).

**Generación de ejemplos con tamaño y composición variable:** para cada
fecha ya construida en el Paso 1, se generan varios subconjuntos
aleatorios de activos, de tamaños distintos (entre 2 y 30). La covarianza
de cada subconjunto es exactamente la submatriz correspondiente de la
Sigma completa ya calculada (no hace falta recalcular nada), y esa
submatriz sigue siendo automáticamente una matriz válida (propiedad del
álgebra lineal).

**Nota sobre el entrenamiento:** como cada ejemplo puede tener un número
distinto de activos, no se pueden agrupar todos en un único lote grande
como en el Paso 3 — se acumula gradiente cada 8 ejemplos (mini-batch, ver
`train_factor.py`) en vez de en un solo lote gigante, lo que sigue
haciendo que cada época tarde más en términos absolutos que el
entrenamiento por lotes completos del Paso 3. Se usa la configuración de
ventanas con mejor equilibrio identificada antes (`history_days=504,
horizon_days=42, step_days=5`).

In [ ]:
from src.predictor.asset_subsets import build_variable_asset_windows
from src.predictor.factor_model import AssetFactorEncoder
from src.predictor.train_factor import train_predictor_factor, predict_sigma_factor

# Reutiliza la configuración de ventanas ya construida en la sección 3.10
# (windows_best, con history_days=504, horizon_days=42, step_days=5)
var_windows = build_variable_asset_windows(
    windows_best, n_assets=n_assets, n_subsets_per_example=5, seed=42,
)
print(f"Ejemplos con tamaño fijo (base): {len(windows_best['X'])}")
print(f"Ejemplos con tamaño variable (expandido): {len(var_windows['X'])}")
print(f"Tamaños de los primeros 10 ejemplos: {[x.shape[1] for x in var_windows['X'][:10]]}")

In [ ]:
# Split temporal con purga, igual que en el Paso 3 (reutilizando la misma lógica,
# aplicada sobre las fechas — el tamaño variable no afecta al criterio temporal)
split_test_var = temporal_split_purged(
    var_windows, test_fraction=0.2,
    history_days=HISTORY_DAYS_BEST, horizon_days=HORIZON_DAYS_BEST, step_days=STEP_DAYS_BEST,
)
windows_train_only_var = {
    "X": split_test_var["X_train"], "y": split_test_var["y_train"], "cutoff_dates": split_test_var["dates_train"],
}
split_val_var = temporal_split_purged(
    windows_train_only_var, test_fraction=0.2,
    history_days=HISTORY_DAYS_BEST, horizon_days=HORIZON_DAYS_BEST, step_days=STEP_DAYS_BEST,
)

X_train_var, y_train_var = split_val_var["X_train"], split_val_var["y_train"]
X_val_var, y_val_var = split_val_var["X_test"], split_val_var["y_test"]
X_test_var, y_test_var = split_test_var["X_test"], split_test_var["y_test"]

print(f"Entrenamiento: {len(X_train_var)} ejemplos")
print(f"Validación:    {len(X_val_var)} ejemplos")
print(f"Test:          {len(X_test_var)} ejemplos")

In [ ]:
# Baseline para esta versión: la covarianza de cada ventana de entrada,
# igual que antes pero calculado ejemplo a ejemplo (tamaños distintos)
baseline_preds_var = baseline_predict(X_test_var)
baseline_mse_var = evaluate_predictions(baseline_preds_var, y_test_var)["mse_medio"]
print(f"Baseline MSE (tamaño variable): {baseline_mse_var:.4e}")

In [ ]:
torch.manual_seed(42)
factor_model = AssetFactorEncoder(n_factors=4, cell_type="gru", hidden_size=16, dropout=0.2)

# Aviso: entrenamiento por mini-batches de 8 ejemplos (no por lotes completos como en
# el Paso 3), puede tardar más que las celdas anteriores.
# verbose_every=50 imprime progreso cada 50 épocas para confirmar que avanza.
# history_csv_path guarda el historial CADA época en disco: si se interrumpe la celda
# (con el botón de stop), el progreso hasta ese punto no se pierde.
factor_result = train_predictor_factor(
    factor_model, X_train_var, y_train_var, X_val_var, y_val_var,
    max_epochs=1000, patience=150, device=DEVICE, verbose_every=50,
    history_csv_path="../data/processed/factor_model_training_history.csv",
)
print(f"Modelo de factores — mejor época: {factor_result['best_epoch']}, val_loss: {factor_result['best_val_loss']:.6e}")

In [ ]:
history_df_factor = pd.DataFrame(factor_result["history"])
history_df_factor.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(10, 4), title="Curva de entrenamiento — modelo de factores (tamaño variable)")
plt.axvline(factor_result["best_epoch"], color="green", linestyle="--", alpha=0.6, label="mejor época")
plt.legend()
plt.show()

history_df_factor.to_csv("../data/processed/factor_model_training_history.csv", index=False)
print("Historial guardado en data/processed/factor_model_training_history.csv")

In [ ]:
factor_preds_test = [predict_sigma_factor(factor_model, x, device=DEVICE) for x in X_test_var]
factor_mse = evaluate_predictions(factor_preds_test, y_test_var)["mse_medio"]

comparison_factor = pd.DataFrame([
    {"modelo": "Baseline", "mse": baseline_mse_var, "mejor_epoca": None},
    {"modelo": "Modelo de factores (tamaño variable)", "mse": factor_mse, "mejor_epoca": factor_result["best_epoch"]},
])
comparison_factor

### Comprobación: la misma red funciona con cualquier número de activos

Se prueba explícitamente sobre un subconjunto de tamaño no usado antes en
el entrenamiento (por ejemplo, todos los 30 activos a la vez), para
confirmar que el modelo generaliza a tamaños distintos sin ningún
reentrenamiento ni ajuste adicional.

In [ ]:
# Predicción con los 30 activos completos (tamaño no necesariamente visto tal cual en entrenamiento)
X_full_example = X_test_b[0] if 'X_test_b' in dir() else windows_best["X"][0]
pred_full = predict_sigma_factor(factor_model, X_full_example, device=DEVICE)
print(f"Predicción con {pred_full.shape[0]} activos — shape: {pred_full.shape}")
print(f"Autovalor mínimo (debe ser >= 0): {np.linalg.eigvalsh(pred_full).min():.4e}")